**Versão 3:** corrige o caminho interno das funções do debootstrap na segunda etapa de instalação do Debian ARM64.

**Versão corrigida:** evita o diretório /root do anfitrião sobrepor o script do Debian no PRoot.

# 🎮 Compilar Plymouth corrigido para R36S (Debian 13 ARM64)
**Clique no botão ▶ da célula abaixo.** O notebook tenta montar um ambiente Debian 13 ARM64 usando QEMU/PRoot, baixa o código oficial `plymouth 24.004.60-5`, aplica as duas correções upstream do bug [2103533](https://bugs.launchpad.net/bugs/2103533) e produz pacotes `.deb` ARM64.

**Atenção:** processo experimental que pode demorar bastante no Colab gratuito. A compilação ainda não foi validada neste ambiente ou no R36S. O notebook não conecta ao console nem altera `uInitrd`, kernel ou partição BOOT. Não instale automaticamente os `.deb` resultantes.

Se falhar, o notebook baixa um arquivo `relatorio_compilacao_plymouth.txt`; envie esse arquivo no ChatGPT para investigação.


In [ ]:
# Execute esta celula pelo botao ▶. O Colab baixa automaticamente o pacote ou o relatorio de erro.
import subprocess, pathlib, shutil
from google.colab import files
pasta = pathlib.Path("/content")
script = pasta / "compilar_plymouth_arm64.sh"
log = pasta / "relatorio_compilacao_plymouth.txt"
saida = pasta / "plymouth-r36s-arm64-pacotes"
script.write_text("#!/usr/bin/env bash\nset -Eeuo pipefail\nexport DEBIAN_FRONTEND=noninteractive\nexport TZ=Etc/UTC\nROOT=/content/r36s-debian-arm64\nOUT=/content/plymouth-r36s-arm64-pacotes\necho '[1/4] Instalando ferramentas no Colab (sem WSL)...'\napt-get update -qq\napt-get install -y -qq --no-install-recommends debootstrap debian-archive-keyring proot qemu-user-static ca-certificates curl >/dev/null\nmkdir -p \"$ROOT\" \"$OUT\"\necho '[2/4] Preparando Debian 13 ARM64 em ambiente isolado (pode demorar)...'\nif [ -x \"$ROOT/debootstrap/debootstrap\" ]; then\n  echo \"Base Debian ja preparada nesta sessao; reutilizando arquivos para poupar tempo.\"\nelse\n  debootstrap --arch=arm64 --foreign --variant=minbase trixie \"$ROOT\" https://deb.debian.org/debian\nfi\nmkdir -p \"$ROOT/usr/bin\" \"$ROOT/usr/local/bin\" \"$ROOT/root\" \"$ROOT/content\"\ncp /usr/bin/qemu-aarch64-static \"$ROOT/usr/bin/\"\ncp -L /etc/resolv.conf \"$ROOT/etc/resolv.conf\" || true\n# Segundo estagio usa as funcoes extraidas na primeira etapa.\nif [ ! -s \"$ROOT/debootstrap/functions\" ]; then\n  if [ -s /usr/share/debootstrap/functions ]; then\n    cp /usr/share/debootstrap/functions \"$ROOT/debootstrap/functions\"\n  else\n    echo \"ERRO: as funcoes do debootstrap nao foram encontradas.\"\n    exit 1\n  fi\nfi\ncat > \"$ROOT/etc/apt/sources.list\" <<'SOURCES'\ndeb http://deb.debian.org/debian trixie main\ndeb-src http://deb.debian.org/debian trixie main\nSOURCES\ncat > \"$ROOT/usr/local/bin/build-r36s.sh\" <<'ARM_SCRIPT'\n#!/bin/bash\nset -Eeuo pipefail\nexport DEBIAN_FRONTEND=noninteractive\nexport TZ=Etc/UTC\necho '[3/4] Configurando Debian ARM64 e dependencias de compilacao...'\ntest -s /debootstrap/functions || { echo \"ERRO: /debootstrap/functions nao encontrado no Debian ARM64\"; exit 1; }\nDEBOOTSTRAP_DIR=/debootstrap /debootstrap/debootstrap --second-stage\ndpkg --print-architecture | grep -qx arm64\napt-get update\napt-get install -y --no-install-recommends ca-certificates curl quilt dpkg-dev build-essential fakeroot file\napt-get build-dep -y plymouth\nmkdir -p /root/plymouth-work\ncd /root/plymouth-work\napt-get source plymouth=24.004.60-5\ncd plymouth-24.004.60\necho 'Aplicando dois patches upstream (LP 2103533)...'\nget_patch() {\n  local commit=\"$1\" output=\"$2\" u\n  for u in \\\n    \"https://gitlab.freedesktop.org/plymouth/plymouth/-/commit/${commit}.patch\" \\\n    \"https://cgit.freedesktop.org/plymouth/patch/?id=${commit}\"; do\n    if curl -fsSL --retry 2 --connect-timeout 15 --max-time 90 \"$u\" -o \"$output\" &&\n      grep -q '^diff --git ' \"$output\"; then\n      echo \"Patch oficial obtido: ${commit}\"\n      return 0\n    fi\n  done\n  echo \"ERRO: nao consegui baixar o patch oficial $commit (verifique a rede do Colab).\" >&2\n  return 1\n}\nget_patch 63597f92d108237a3ab7d2343a602a95edddd4e5 debian/patches/0100-r36s-null-terminal.patch\nget_patch 5c10072a978dd7566559f44a54c3e031bb4cb216 debian/patches/0101-r36s-null-renderer.patch\nprintf '\\n0100-r36s-null-terminal.patch\\n0101-r36s-null-renderer.patch\\n' >> debian/patches/series\nQUILT_PATCHES=debian/patches quilt push -a\npython3 - <<'PY'\nfrom pathlib import Path\np=Path('debian/changelog')\nheader='''plymouth (24.004.60-5+r36s1) trixie; urgency=medium\n\n  * Backport upstream null-terminal guards, LP: #2103533.\n  * Build para teste no R36S (nao instalado).\n\n -- R36S Plymouth Build <noreply@example.invalid>  Sat, 31 Oct 2026 15:00:00 +0000\n\n'''\np.write_text(header+p.read_text())\nPY\necho '[4/4] Compilando pacotes nativos Debian ARM64 (pode levar bastante tempo)...'\nDEB_BUILD_OPTIONS=nocheck dpkg-buildpackage -b -us -uc -j2\nfind /root/plymouth-work -maxdepth 1 -name '*_arm64.deb' -exec cp -v {} /content/plymouth-r36s-arm64-pacotes/ \\;\ntest -f /content/plymouth-r36s-arm64-pacotes/plymouth_24.004.60-5+r36s1_arm64.deb\ntest -f /content/plymouth-r36s-arm64-pacotes/libplymouth5_24.004.60-5+r36s1_arm64.deb\nfor f in /content/plymouth-r36s-arm64-pacotes/*.deb; do\n  dpkg-deb --field \"$f\" Package Version Architecture\ndone\nsha256sum /content/plymouth-r36s-arm64-pacotes/*.deb > /content/plymouth-r36s-arm64-pacotes/SHA256SUMS.txt\necho 'COMPILACAO_CONCLUIDA_ARM64'\nARM_SCRIPT\nchmod +x \"$ROOT/usr/local/bin/build-r36s.sh\"\necho 'Iniciando emulacao ARM64 via QEMU/PRoot. Nao altera arquivos do R36S.'\ntest -s \"$ROOT/usr/local/bin/build-r36s.sh\" || { echo 'ERRO: script ARM64 nao foi criado.'; exit 1; }\nproot -0 -r \"$ROOT\" -b /dev -b /proc -b /sys -b /content:/content -b /etc/resolv.conf:/etc/resolv.conf -q /usr/bin/qemu-aarch64-static -w /root /bin/bash /usr/local/bin/build-r36s.sh\n", encoding="utf-8")
script.chmod(0o755)
print("Iniciando compilacao Plymouth Debian 13 para ARM64.")
print("Usa emulacao QEMU; pode levar bastante tempo no Colab gratuito.\n")
with log.open("w", encoding="utf-8") as arquivo:
    p = subprocess.Popen(["bash", str(script)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    linhas = 0
    ultimas = []
    for linha in p.stdout:
        arquivo.write(linha)
        arquivo.flush()
        linhas += 1
        ultimas.append(linha.rstrip())
        ultimas = ultimas[-22:]
        if linha.startswith("[") or "ERRO:" in linha or "COMPILACAO_CONCLUIDA" in linha or linhas % 80 == 0:
            print(linha.rstrip()[:400])
    codigo = p.wait()
print("\nCodigo de saida:", codigo)
if codigo == 0 and list(saida.glob("*.deb")):
    arquivo_zip = shutil.make_archive(str(pasta / "Plymouth_R36S_ARM64_Corrigido"), "zip", root_dir=str(saida))
    print("Pacotes compilados. A compilacao ainda NAO comprova funcionamento no R36S.")
    print("BAIXANDO:", arquivo_zip)
    files.download(arquivo_zip)
else:
    print("Compilacao nao concluida. Final do relatorio:")
    print("\n".join(ultimas))
    print("Baixando relatorio para corrigirmos o problema, sem tocar no console.")
    files.download(str(log))
